# miniLLM 49M — SFT v3.1

**Une seule variable change par rapport au SFT v3 : les données** (mêmes hyperparamètres : 2 époques, LR 1e-4 → 1e-5, wd 0.1, dropout 0.1). Ainsi, si les résultats bougent, c'est grâce aux données.

- **+8 000 exemples synthétiques** calculés par code (additions/soustractions/tables, petits problèmes avec le calcul écrit avant le résultat, suites, intrus, logique, langue, définitions). Aucune de tes 337 questions de test n'y figure.
- **French Alpaca réduit** (15 000 → 8 000) ; **réponses GPT-4 corrigées** : le chargeur ignorait le format `conversations`, donc la source n'avait rien chargé dans le v3.
- **Nouvelle mesure `skills_eval.py`** : calculs et problèmes chiffrés **tenus à l'écart de l'entraînement** (comparaison honnête v3 ↔ v3.1).
- Écrit dans `checkpoints/sft_v3_1` : rien n'est remplacé.

**Utilisation :** Exécution → type d'exécution → GPU (T4), puis **Tout exécuter**. Si la session coupe, relance.

In [ ]:
# ── 1) Montage Drive, code, garde-fou, courbe du SFT v3 ──
import os, sys, glob, shutil, json
from google.colab import drive
drive.mount("/content/drive")

cands = (["/content/drive/MyDrive/MiniLLM_v2"] + glob.glob("/content/drive/MyDrive/*/MiniLLM_v2")
         + glob.glob("/content/drive/Shareddrives/*/MiniLLM_v2"))
BASE = next((c for c in cands if os.path.isdir(f"{c}/data")), None)
if BASE is None:
    raise SystemExit("Dossier MiniLLM_v2 introuvable (Drive > Partagés > appui long > Ajouter un raccourci > Mon Drive).")
DATA, PT_OUT = f"{BASE}/data", f"{BASE}/checkpoints/pretrain"
SFT_OUT = f"{BASE}/checkpoints/sft_v3_1"           # fixe : une relance reprend ce run
OLD_OUT = f"{BASE}/checkpoints/sft_v3"
CODE_DIR = "/content/minillm_v2"

!pip -q install tokenizers datasets
if os.path.exists(f"{CODE_DIR}/.git"):
    !git -C {CODE_DIR} fetch -q origin v2.1.1 && git -C {CODE_DIR} checkout -q v2.1.1 && git -C {CODE_DIR} pull -q --ff-only
else:
    !git clone -q -b v2.1.1 https://github.com/bono-p/minillm_v2.git {CODE_DIR}
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)

import torch
assert os.path.exists(f"{PT_OUT}/best.pt"), f"{PT_OUT}/best.pt introuvable"
b = torch.load(f"{PT_OUT}/best.pt", map_location="cpu", weights_only=False)
print(f"Base du SFT : pré-entraînement it={b['iter']} | val_loss={b['val_loss']:.4f}")
if int(b["iter"]) < 49_000:
    raise SystemExit("Le pré-entraînement n'est pas terminé (it < 49 000).")
del b

def courbe(run_dir, label):
    """Affiche la courbe val_loss d'un run SFT (lue dans log.jsonl) : où est le minimum, remonte-t-elle ensuite ?"""
    path = f"{run_dir}/log.jsonl"
    if not os.path.exists(path):
        print(f"({label} : pas de log.jsonl)"); return
    ev = {}
    for l in open(path):
        if l.strip():
            e = json.loads(l)
            if e.get("type") == "eval":
                ev[e["it"]] = e["val_loss"]
    if not ev:
        print(f"({label} : aucune ligne eval)"); return
    its = sorted(ev); best = min(its, key=lambda i: ev[i])
    print(f"\n── {label} : val_loss par itération (minimum à it={best} : {ev[best]:.4f}) ──")
    for i in its:
        print(f"  it {i:>5}  val {ev[i]:.4f}" + ("   ← minimum" if i == best else ""))
    print(f"  dernière : it {its[-1]} val {ev[its[-1]]:.4f}  (écart avec le minimum : {ev[its[-1]] - ev[best]:+.4f})")

courbe(OLD_OUT, "SFT v3 (déjà entraîné)")


In [ ]:
# ── 2) Données SFT v3.1 (synthetic_plus + French Alpaca réduit + GPT-4 FR corrigé) ──
SFT_DATA = f"{DATA}/sft_v3_1"
COMMON = f"--tokenizer {DATA}/tokenizer.json --out {SFT_DATA} --oasst 3000 --piaf 2000 --evol 0 --sharegpt 0 --persona personnalite.jsonl --persona_repeat 8 --plus 1"
if os.path.exists(f"{SFT_DATA}/meta.json"):
    print("✓ Données SFT v3.1 déjà prêtes :", json.load(open(f"{SFT_DATA}/meta.json"))["sources"])
else:
    !python sft_data.py {COMMON} --alpaca 8000 --alpaca_gpt4 8000
    srcs = json.load(open(f"{SFT_DATA}/meta.json"))["sources"]
    if "alpaca_gpt4_fr" not in srcs:
        print("\n⚠️  Les réponses GPT-4 FR n'ont rien chargé : repli sur 15 000 French Alpaca (comme le v3).")
        shutil.rmtree(SFT_DATA)
        !python sft_data.py {COMMON} --alpaca 15000 --alpaca_gpt4 0
    print("\nSources finales :", json.load(open(f"{SFT_DATA}/meta.json"))["sources"])

LOCAL = "/content/minillm_local"
os.makedirs(LOCAL, exist_ok=True)
if os.path.exists(f"{LOCAL}/sft_v3_1"):
    shutil.rmtree(f"{LOCAL}/sft_v3_1")
shutil.copytree(SFT_DATA, f"{LOCAL}/sft_v3_1")
shutil.copy(f"{DATA}/tokenizer.json", f"{LOCAL}/tokenizer.json")


In [ ]:
# ── 3) Entraînement : mêmes hyperparamètres que le v3 (seules les données changent) ──
!python train.py --mode sft --data_dir {LOCAL}/sft_v3_1 --out_dir {SFT_OUT} --init_from {PT_OUT}/best.pt \
    --epochs 2 --lr 1e-4 --min_lr 1e-5 --weight_decay 0.1 --dropout 0.1 --warmup_iters 100 --patience 0 \
    --eval_every 250 --eval_every_dense 100 --eval_dense_until 1500 --save_every 250
courbe(SFT_OUT, "SFT v3.1 (nouveau)")


In [ ]:
# ── 4) Comparaison v3 ↔ v3.1 sur les MÊMES données de validation et les MÊMES problèmes tenus à l'écart ──
OLD, NEW = f"{OLD_OUT}/best.pt", f"{SFT_OUT}/best.pt"
OLD_VAL = f"{DATA}/sft_v3"            # jeu de validation du v3 (jamais vu à l'entraînement par aucun des deux)
for name, ck in [("SFT v3 (avant)", OLD), ("SFT v3.1 (nouveau)", NEW)]:
    if not os.path.exists(ck):
        print(f"\n(pas de {ck})"); continue
    print("\n" + "=" * 72 + f"\n{name}\n" + "=" * 72)
    print("— compétences chiffrées (problèmes tenus à l'écart de l'entraînement) —")
    !python skills_eval.py --ckpt {ck} --n 60
    print("— QA sur la validation du v3 (même jeu pour les deux) —")
    !python evaluate.py qa --ckpt {ck} --sft_dir {OLD_VAL} --n 500

# 5) Persona, démo et 337 questions ouvertes sur le nouveau modèle
!python evaluate.py persona --ckpt {NEW}
!python evaluate.py demo    --ckpt {NEW}
!python evaluate.py openqa  --ckpt {NEW} --n 1000
